In [ ]:
!pip install selenium
!apt-get update # to update apt repositories
!apt install -y chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.7/481.7 kB 23.4 MB/s eta 0:00:00
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,192 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,454 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchp

In [ ]:
# Install necessary libraries and dependencies
!pip install selenium

# Install Chromium and its dependencies
!apt-get update -qq # Update apt repositories
!apt install -y chromium-browser # Install Chromium browser
!apt install -y libx11-xcb1 libgdk-pixbuf2.0-0 libnss3 libgconf-2-4 libasound2 libatk-1.0-0 libatk-bridge2.0-0 libxcomposite1 libxrandr2

# Download the correct version of chromedriver matching the chromium version
!wget https://chromedriver.storage.googleapis.com/114.0.5735.90/chromedriver_linux64.zip
!unzip chromedriver_linux64.zip
!mv chromedriver /usr/bin/chromedriver

# Set up environment variables
import os
os.environ["PATH"] += ":/usr/lib/chromium-browser/chromedriver"  # Add chromedriver to PATH

# Set up the environment for Selenium to work with headless Chrome
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time

# Set up Chrome options for headless mode
chrome_options = Options()
chrome_options.add_argument('--headless')  # Ensure no GUI is opened
chrome_options.add_argument('--no-sandbox')  # Needed for Chrome to run in restricted environments like Colab
chrome_options.add_argument('--disable-dev-shm-usage')  # Overcome limited shared memory issue in Colab
chrome_options.add_argument('--remote-debugging-port=9222')  # Allow remote debugging
chrome_options.add_argument('--disable-gpu')  # Disable GPU acceleration in headless mode
chrome_options.add_argument('--disable-software-rasterizer')  # Disable software rasterizer for better performance
chrome_options.add_argument('--single-process')  # Ensure Chrome runs in a single process in headless mode

# Specify the path to the chromedriver
chrome_driver_path = '/usr/bin/chromedriver'

# Check if chromedriver is correctly installed and accessible
print(f"ChromeDriver path: {chrome_driver_path}")

# Initialize the WebDriver
driver = webdriver.Chrome(service=Service(chrome_driver_path), options=chrome_options)

# Open Google Drive folder (Make sure it's shared publicly or you are logged in)
drive_url = 'https://drive.google.com/drive/folders/1nkI7lddegUWDvmGsmn68QkB5ikp5_3kt'  # Replace with your folder ID
driver.get(drive_url)

# Wait for the page to load
time.sleep(5)

# Find the list of files in the folder
file_names = []
file_elements = driver.find_elements_by_xpath('//div[@role="link"]//div[@class="Q5txwe"]')  # XPath for file names

# Extract file names
for file in file_elements:
    file_names.append(file.text)

# Print out the list of file names
print("List of file names:")
for file_name in file_names:
    print(file_name)

# Close the driver
driver.quit()


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
chromium-browser is already the newest version (1:85.0.4183.83-0ubuntu2.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package libatk-1.0-0
E: Couldn't find any package by glob 'libatk-1.0-0'
--2024-12-09 10:02:37--  https://chromedriver.storage.googleapis.com/114.0.5735.90/chromedriver_linux64.zip
Resolving chromedriver.storage.googleapis.com (chromedriver.storage.googleapis.com)... 173.194.212.207, 173.194.210.207, 173.194.215.207, ...
Connecting to chromedriver.storage.googleapis.com (chromedriver.storage.googleapis.com)|173.194.212.207|:443... connected.
HTTP 

In [ ]:
pip install tqdm

In [ ]:

import io
import os
import google.auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload
from tqdm import tqdm  # Importing the tqdm library for progress bar
from google.colab import auth  # This is specific to Google Colab
from google.auth.transport.requests import Request  # To refresh credentials if needed

def download_file(real_file_id, destination_path):
    """Downloads a file from Google Drive and saves it to a specified location.

    Args:
        real_file_id (str): ID of the file to download.
        destination_path (str): The path where the file should be saved.

    Returns:
        str: The path to the downloaded file, or None if an error occurs.
    """
    try:
        # Authenticate and create the API client for Google Drive in Colab
        auth.authenticate_user()  # This is specific to Google Colab
        creds, _ = google.auth.default()

        # Create Google Drive API client
        service = build("drive", "v3", credentials=creds)

        # Request to get file
        request = service.files().get_media(fileId=real_file_id)

        # Get the file's metadata to determine its size for the progress bar
        file_metadata = service.files().get(fileId=real_file_id).execute()
        file_size = int(file_metadata.get("size", 0))

        # Initialize the tqdm progress bar
        with open(destination_path, "wb") as file:
            downloader = MediaIoBaseDownload(file, request)
            done = False
            pbar = tqdm(total=file_size, unit="B", unit_scale=True, desc="Downloading")  # Initialize tqdm

            while not done:
                status, done = downloader.next_chunk()
                if status:
                    pbar.update(int(status.resumable_progress))  # Update the progress bar
                    tqdm.write(f"Download {int(status.progress() * 100)}%")  # Print the progress

            pbar.close()  # Close the progress bar when done

        print(f"File downloaded to {destination_path}")

    except HttpError as error:
        print(f"An error occurred: {error}")
        return None
    except Exception as error:
        print(f"Unexpected error: {error}")
        return None

    return destination_path

if __name__ == "__main__":
    # Example file ID (replace with the actual file ID you want to download)
    file_id = "1jwmzfV_8s0kSfFPRej6i_HOmMK-t4PZH"

    # Destination path for the downloaded file (e.g., "my_file.pdf")
    destination_path = "downloaded_file.mp4"

    # Call the function to download the file
    downloaded_file_path = download_file(real_file_id=file_id, destination_path=destination_path)

    if downloaded_file_path:
        print(f"Download completed. File saved at: {downloaded_file_path}")
    else:
        print("Download failed.")

Downloading: 1.29MB [00:00, 1.58MB/s]

Download 100%
File downloaded to downloaded_file.mp4
Download completed. File saved at: downloaded_file.mp4


In [ ]:
import io
import os
import google.auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload
from tqdm import tqdm  # Importing the tqdm library for progress bar
from google.colab import auth  # Specific to Google Colab
from google.auth.transport.requests import Request  # To refresh credentials if needed

# Function to download files
def download_file(file_id, destination_path):
    """Downloads a file from Google Drive and saves it to a specified location.

    Args:
        file_id (str): ID of the file to download.
        destination_path (str): The path where the file should be saved.

    Returns:
        str: The path to the downloaded file, or None if an error occurs.
    """
    try:
        # Authenticate and create the API client for Google Drive in Colab
        auth.authenticate_user()  # Colab specific authentication
        creds, _ = google.auth.default()

        # Create Google Drive API client
        service = build("drive", "v3", credentials=creds)

        # Request to get file metadata
        request = service.files().get_media(fileId=file_id)

        # Get the file's metadata to determine its size for the progress bar
        file_metadata = service.files().get(fileId=file_id).execute()
        file_size = int(file_metadata.get("size", 0))

        # Ensure that the destination folder exists
        os.makedirs(os.path.dirname(destination_path), exist_ok=True)

        # Initialize the tqdm progress bar
        with open(destination_path, "wb") as file:
            downloader = MediaIoBaseDownload(file, request)
            done = False
            pbar = tqdm(total=file_size, unit="B", unit_scale=True, desc="Downloading")  # Initialize tqdm

            while not done:
                status, done = downloader.next_chunk()
                if status:
                    pbar.update(int(status.resumable_progress))  # Update the progress bar
                    tqdm.write(f"Download {int(status.progress() * 100)}%")  # Print the progress

            pbar.close()  # Close the progress bar when done

        print(f"File downloaded to {destination_path}")

    except HttpError as error:
        print(f"An error occurred: {error}")
        return None
    except Exception as error:
        print(f"Unexpected error: {error}")
        return None

    return destination_path

# Function to list files from Google Drive folder and filter by file names
def get_files_from_folder(folder_id, file_names):
    """List files in a Google Drive folder and filter by file names.

    Args:
        folder_id (str): ID of the folder to list files from.
        file_names (list): List of file names to search for.

    Returns:
        list: A list of file IDs for the matching files.
    """
    try:
        # Authenticate and create the API client for Google Drive
        auth.authenticate_user()  # Colab specific authentication
        creds, _ = google.auth.default()

        service = build("drive", "v3", credentials=creds)

        # List files in the folder and handle pagination
        query = f"'{folder_id}' in parents"
        results = []
        page_token = None
        file_count = 0

        while True:
            # Make API request to list files
            print(f"Fetching page {file_count + 1}...")  # Debugging page fetch
            files_result = service.files().list(
                q=query,
                fields="nextPageToken, files(id, name)",
                pageToken=page_token,
                pageSize=1000  # Increase page size (max is 1000)
            ).execute()

            # Add files from the current page to the results
            files = files_result.get('files', [])
            results.extend(files)
            file_count += len(files)
            print(f"Fetched {len(files)} files, total: {file_count}")

            # Check if there are more files (pagination)
            page_token = files_result.get('nextPageToken', None)
            if not page_token:
                print("No more pages left.")
                break  # Exit loop if there are no more pages

        # Debug print: Output the total number of files found
        print(f"Found {file_count} files in total.")

        # Filter files by the given file names
        matched_files = []
        for file in results:
            # Perform case-insensitive match (ignores case) and strip extra spaces
            if any(file_name.strip().lower() == file['name'].strip().lower() for file_name in file_names):
                matched_files.append(file)

        return matched_files

    except HttpError as error:
        print(f"An error occurred: {error}")
        return []
    except Exception as error:
        print(f"Unexpected error: {error}")
        return []

# Function to extract folder ID from Google Drive URL
def extract_folder_id_from_url(url):
    """Extract folder ID from a Google Drive folder URL.

    Args:
        url (str): The URL of the Google Drive folder.

    Returns:
        str: The folder ID.
    """
    # Google Drive folder URL format: https://drive.google.com/drive/folders/<folder_id>
    folder_id = url.split("/")[-1]
    return folder_id

# Main execution block
if __name__ == "__main__":
    # Google Drive folder URL (replace with your folder link)
    folder_url = "https://drive.google.com/drive/folders/1S-G5qoWth0aI9fLzKs8Myv2LyMrq8jUh"

    # List of file names to download (replace with your list of filenames)
    file_names = ["Book.mp4", "Boy.mp4", "Baby.mp4", "Brother.mp4", "Bad.mp4", "Back.mp4", "Bye.mp4"]

    # Extract the folder ID from the URL
    folder_id = extract_folder_id_from_url(folder_url)

    # Get list of files from the folder
    files_to_download = get_files_from_folder(folder_id, file_names)

    # Check if any files are found
    if files_to_download:
        # Download each file
        for file in files_to_download:
            print(f"Downloading {file['name']}...")
            destination_path = os.path.join("/content/drive/MyDrive/B", file['name'])
            download_file(file["id"], destination_path)
            print(f"Downloaded {file['name']} to {destination_path}")
    else:
        print("No files found for the specified names.")


Fetching page 1...
Fetched 521 files, total: 521
No more pages left.
Found 521 files in total.


Downloading: 2.43MB [00:02, 1.09MB/s]


Download 100%
File downloaded to /content/drive/MyDrive/B/Teacher.mp4
Downloaded Teacher.mp4 to /content/drive/MyDrive/B/Teacher.mp4


Downloading: 3.09MB [00:01, 1.79MB/s]


Download 100%
File downloaded to /content/drive/MyDrive/B/Tea.mp4
Downloaded Tea.mp4 to /content/drive/MyDrive/B/Tea.mp4


...

Download 100%
File downloaded to /content/drive/MyDrive/B/Today.mp4
Downloaded Today.mp4 to /content/drive/MyDrive/B/Today.mp4


Downloading: 2.70MB [00:01, 1.66MB/s]


Download 100%
File downloaded to /content/drive/MyDrive/B/That.mp4
Downloaded That.mp4 to /content/drive/MyDrive/B/That.mp4


Downloading: 1.56MB [00:01, 1.06MB/s]

Download 100%
File downloaded to /content/drive/MyDrive/B/There.mp4
Downloaded There.mp4 to /content/drive/MyDrive/B/There.mp4


In [ ]:
pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.1/36.1 MB 44.8 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import mediapipe as mp
import zipfile
import shutil

# Initialize MediaPipe Hands and Pose modules
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

# Specify the direct path to your zip file (update the path below)
zip_filename = '/content/folder.zip'  # Update this path with the actual zip file location
zip_extract_folder = '/content/input_images'  # Folder where images will be extracted

# Create output folder for processed images if it doesn't exist
output_folder = '/content/processed_images'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Unzip the file
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(zip_extract_folder)

# Get list of image files in the extracted folder
image_files = [f for f in os.listdir(zip_extract_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]

# Initialize MediaPipe Hands and Pose models
hands = mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5)
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

# Function to process and save images
def process_images():
    for image_file in image_files:
        # Read image
        img_path = os.path.join(zip_extract_folder, image_file)
        image = cv2.imread(img_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Apply hand detection
        results_hands = hands.process(image_rgb)
        if results_hands.multi_hand_landmarks:
            for hand_landmarks in results_hands.multi_hand_landmarks:
                mp_draw.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        # Apply pose detection
        results_pose = pose.process(image_rgb)
        if results_pose.pose_landmarks:
            mp_draw.draw_landmarks(image, results_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS)

        # Save the processed image
        output_img_path = os.path.join(output_folder, image_file)
        cv2.imwrite(output_img_path, image)
        print(f'Processed and saved: {image_file}')

# Run the function to process all images in the folder
process_images()

# Zip the processed images for easy download
shutil.make_archive('/content/processed_images_zip', 'zip', output_folder)

# Provide a download link for the processed images
from google.colab import files
files.download('/content/processed_images_zip.zip')


BadZipFile: File is not a zip file